In [1]:
import pandas as pd


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report




In [2]:
train_df = pd.read_csv("/home/student/Desktop/python/customer_service_intent_classifier/datasets/Bitext_Sample_Customer_Service_Training_Dataset.csv")
validation_df = pd.read_csv("/home/student/Desktop/python/customer_service_intent_classifier/datasets/Bitext_Sample_Customer_Service_Validation_Dataset.csv")
test_df = pd.read_csv("/home/student/Desktop/python/customer_service_intent_classifier/datasets/Bitext_Sample_Customer_Service_Testing_Dataset.csv")
df = pd.concat([train_df, validation_df, test_df], ignore_index=True)

In [3]:


print("Dataset shape:", df.shape)

print("\nFirst five rows:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

Dataset shape: (8175, 4)

First five rows:
                                           utterance        intent category  \
0   would it be possible to cancel the order I made?  cancel_order    ORDER   
1                                   cancelling order  cancel_order    ORDER   
2  I need assistance canceling the last order I h...  cancel_order    ORDER   
3            problem with canceling the order I made  cancel_order    ORDER   
4        I don't know how to cancel the order I made  cancel_order    ORDER   

  tags  
0  BIP  
1   BK  
2    B  
3    B  
4    B  

Column names:
['utterance', 'intent', 'category', 'tags']

Data types:
utterance    object
intent       object
category     object
tags         object
dtype: object


In [4]:
print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nNumber of unique intents:", df["intent"].nunique())

print("\nIntent distribution:")
print(df["intent"].value_counts())


Missing values:
utterance    0
intent       0
category     0
tags         0
dtype: int64

Duplicate rows: 0

Number of unique intents: 27

Intent distribution:
intent
check_invoice               324
get_invoice                 324
payment_issue               323
review                      315
track_refund                308
set_up_shipping_address     307
place_order                 306
cancel_order                305
track_order                 305
change_order                304
check_refund_policy         302
delivery_options            302
delivery_period             301
contact_customer_service    299
recover_password            298
check_cancellation_fee      298
create_account              298
delete_account              298
complaint                   298
contact_human_agent         297
check_payment_methods       297
change_shipping_address     297
registration_problems       296
newsletter_subscription     295
edit_account                294
get_refund                  294


In [5]:
df = df.dropna(subset=["utterance", "intent"]).copy()

df["utterance"] = df["utterance"].astype(str).str.strip()
df["intent"] = df["intent"].astype(str).str.strip()

df = df[
    (df["utterance"] != "") &
    (df["intent"] != "")
]

df = df.drop_duplicates(subset=["utterance", "intent"])

print("Rows after cleaning:", len(df))

Rows after cleaning: 8175


In [6]:
X = train_df["utterance"]
y = train_df["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Total rows:", len(df))
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Total rows: 8175
Training rows: 5231
Testing rows: 1308


In [7]:
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training feature shape:", X_train_tfidf.shape)
print("Testing feature shape:", X_test_tfidf.shape)

Training feature shape: (5231, 2760)
Testing feature shape: (1308, 2760)


In [8]:
model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Logistic Regression training completed!")

Logistic Regression training completed!


In [9]:
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("Model accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

Model accuracy: 99.39 %

Classification report:
                          precision    recall  f1-score   support

            cancel_order       0.98      0.98      0.98        49
            change_order       1.00      0.98      0.99        46
 change_shipping_address       1.00      1.00      1.00        45
  check_cancellation_fee       0.98      1.00      0.99        49
           check_invoice       1.00      1.00      1.00        52
   check_payment_methods       1.00      1.00      1.00        49
     check_refund_policy       0.98      1.00      0.99        48
               complaint       1.00      1.00      1.00        50
contact_customer_service       1.00      1.00      1.00        51
     contact_human_agent       1.00      1.00      1.00        45
          create_account       0.98      0.96      0.97        47
          delete_account       1.00      0.98      0.99        47
        delivery_options       1.00      1.00      1.00        47
         delivery_period   

In [19]:
print("\n--- Customer Intent Prediction System ---")

while True:

    message = input(
        "\nEnter your customer message (or type 'exit' to stop): "
    )

    if message.strip().lower() == "exit":
        print("Program stopped.")
        break

    if not message.strip():
        print("Please enter a valid message.")
        continue

    # Convert user message into TF-IDF features
    message_tfidf = tfidf.transform([message])

    # Predict intent
    prediction = model.predict(message_tfidf)[0]

    # Display prediction
    print("Customer Message:", message)
    print("Predicted Intent:", prediction)


--- Customer Intent Prediction System ---



Enter your customer message (or type 'exit' to stop):  exit


Program stopped.


In [20]:
import streamlit as st
import sklearn
import pandas as pd

print("Streamlit version:", st.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Pandas version:", pd.__version__)

Streamlit version: 1.65.0
Scikit-learn version: 1.7.2
Pandas version: 2.3.3


In [21]:
from pathlib import Path

PROJECT_DIR = Path.home() / "Desktop" / "python"
DATASET_DIR = PROJECT_DIR / "archive (1)"

print("Project exists:", PROJECT_DIR.exists())
print("Dataset folder exists:", DATASET_DIR.exists())

print("\nFiles inside archive:")
for file in DATASET_DIR.iterdir():
    print(file.name)

Project exists: True
Dataset folder exists: True

Files inside archive:


In [22]:
from pathlib import Path

PROJECT_DIR = Path.home() / "Desktop" / "python"

app_path = PROJECT_DIR / "app.py"

print("Streamlit file will be created at:", app_path)

Streamlit file will be created at: /home/student/Desktop/python/app.py


In [23]:
from pathlib import Path

project_dir = Path.home() / "Desktop" / "python"

app_code = r'''
import streamlit as st
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from pathlib import Path

st.set_page_config(
    page_title="Customer Intent Classification",
    page_icon="💬"
)

st.title("Customer Service Intent Classification")
st.write("Enter a customer message to predict its intent.")


@st.cache_resource
def load_model():

    # Dataset folder
    dataset_dir = Path(__file__).parent / "archive (1)"

    # Find CSV files
    files = list(dataset_dir.glob("*.csv"))

    train_file = next(
        (f for f in files if "training" in f.name.lower()), None
    )
    validation_file = next(
        (f for f in files if "validation" in f.name.lower()), None
    )
    test_file = next(
        (f for f in files if "testing" in f.name.lower()), None
    )

    if not all([train_file, validation_file, test_file]):
        raise FileNotFoundError(
            "Check that archive (1) contains the training, "
            "validation and testing CSV files."
        )

    # Read and concatenate datasets
    train_df = pd.read_csv(train_file)
    validation_df = pd.read_csv(validation_file)
    test_df = pd.read_csv(test_file)

    df = pd.concat(
        [train_df, validation_df, test_df],
        ignore_index=True
    )

    # Clean data
    df = df.dropna(subset=["utterance", "intent"]).copy()

    df["utterance"] = df["utterance"].astype(str).str.strip()
    df["intent"] = df["intent"].astype(str).str.strip()

    df = df[
        (df["utterance"] != "") &
        (df["intent"] != "")
    ]

    df = df.drop_duplicates(subset=["utterance", "intent"])

    X = df["utterance"]
    y = df["intent"]

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    # Convert text into numerical features
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        sublinear_tf=True
    )

    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)

    # Train one model
    model = LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42
    )

    model.fit(X_train_tfidf, y_train)

    # Evaluate
    predictions = model.predict(X_test_tfidf)

    accuracy = accuracy_score(y_test, predictions)

    return vectorizer, model, accuracy, len(df)


try:

    with st.spinner("Training the model... Please wait."):

        vectorizer, model, accuracy, total_records = load_model()

    st.success("Model trained successfully!")

    col1, col2 = st.columns(2)

    col1.metric("Model Accuracy", f"{accuracy * 100:.2f}%")
    col2.metric("Dataset Records", f"{total_records:,}")

    st.divider()

    message = st.text_area(
        "Customer Message",
        placeholder="Example: I want to reset my password"
    )

    if st.button("Predict Intent", type="primary"):

        if not message.strip():

            st.warning("Please enter a customer message.")

        else:

            message_tfidf = vectorizer.transform([message])

            prediction = model.predict(message_tfidf)[0]

            probabilities = model.predict_proba(message_tfidf)[0]

            confidence = max(probabilities) * 100

            st.subheader("Prediction Result")

            st.success(f"Predicted Intent: {prediction}")

            st.write(f"Model confidence: {confidence:.2f}%")

            st.caption(
                "Confidence is an estimated model probability, "
                "not a guarantee of correctness."
            )

except Exception as error:

    st.error(f"Application error: {error}")
'''

app_file = project_dir / "app.py"

app_file.write_text(app_code, encoding="utf-8")

print("Updated app.py successfully!")
print("Location:", app_file)
print("File exists:", app_file.exists())

Updated app.py successfully!
Location: /home/student/Desktop/python/app.py
File exists: True


In [24]:
%cd ~/Desktop/python


/home/student/Desktop/python


In [ ]:
!streamlit run app.py

2026-10-14 12:16:34.564 Uvicorn server started on :::8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://192.168.1.8:8502

  Help agents write better Streamlit apps?
  Install the official Streamlit skills by running streamlit skills in your terminal.

